# 08 · Master Pipeline Kaggle – Zero-Disk Output

Pipeline completo com **isolamento de I/O em memória RAM (tmpfs)**: prepara scripts a partir do checkout do GitHub, detecta GPU, configura Google Drive SOMENTE para persistência/backup de modelos, instala ComfyUI e sobe o servidor com `--output-directory /dev/shm/comfy_ui_output` e `--temp-directory /dev/shm/comfy_ui_temp`.

**Modelos do Dataset:** lidos diretamente do mount somente-leitura em `/kaggle/input/<slug>` (anexado via "Add Data" na UI do Kaggle), sem download/cópia para o SSD. O `extra_model_paths.yaml` registra essa raiz como `dataset_models`.

**Garantia de não-persistência em disco:** nenhuma imagem, preview ou latente toca `/kaggle/working` — evasão total de IOPS no diretório monitorado pelo snapshot/versionamento do Kaggle. O único artefato gravado no diretório persistente é `output_secure.zip`, produzido pela célula de gerenciamento de ciclo de vida de artefatos (polling da API → compactação → purga da RAM volátil).

In [ ]:
from pathlib import Path
import subprocess, sys, os, time, json

REPO_URL = "https://github.com/automadevs/colab-pipeline.git"
WORKDIR = Path("/kaggle/working")
REPO_DIR = WORKDIR / "colab-pipeline"
SCRIPTS_DIR = WORKDIR / "scripts"
COMFYUI_DIR = WORKDIR / "ComfyUI"

# Zero-Disk Output: todo I/O de geração isolado em tmpfs (RAM volátil).
# /dev/shm não é varrido pelo snapshot/versionamento do Kaggle.
SHM_OUTPUT = Path("/dev/shm/comfy_ui_output")
SHM_TEMP = Path("/dev/shm/comfy_ui_temp")
OUTPUT_DIR = SHM_OUTPUT
SECURE_ZIP = WORKDIR / "output_secure.zip"

# Modelos locais: diretório gravável no SSD (para a célula Civitai -> Local e custom nodes).
MODELS_DIR = COMFYUI_DIR / "models"

def _get_secret(name):
    value = os.environ.get(name)
    if value:
        return value
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return None

def resolve_dataset_name(override=None):
    """Resolve o dataset Kaggle: override explícito ou KAGGLE_USERNAME/KAGGLE_DATASET_NAME."""
    if override:
        return override
    username = _get_secret("KAGGLE_USERNAME")
    dataset_name = _get_secret("KAGGLE_DATASET_NAME")
    if not username or not dataset_name:
        raise ValueError(
            "Dataset Kaggle não resolvido. Configure os Secrets do Kaggle:\n"
            "  - KAGGLE_USERNAME: seu username Kaggle\n"
            "  - KAGGLE_DATASET_NAME: nome do dataset (ex: comfydocs)\n"
            "Ou defina DATASET_OVERRIDE no início desta célula."
        )
    return f"{username}/{dataset_name}"

DATASET_OVERRIDE = None  # ex: "meuusuario/meudataset" para forçar
DATASET = resolve_dataset_name(DATASET_OVERRIDE)
print(f"[INFO] Dataset alvo: {DATASET}")

DRIVE_BASE = "Automa/ComfyUI"

In [ ]:
# Dataset anexado como INPUT do notebook (Add Data na UI do Kaggle) -- leitura direta,
# sem download/cópia para o SSD. O Colab publica os modelos já nas subpastas de
# categoria corretas, então o ComfyUI lê o Dataset montado sem nenhuma cópia extra.
DATASET_SLUG = _get_secret("KAGGLE_DATASET_NAME")
if not DATASET_SLUG:
    raise RuntimeError(
        "Secret KAGGLE_DATASET_NAME não configurado -- necessário para localizar "
        "o mount do dataset em /kaggle/input."
    )

DATASET_INPUT_DIR = Path("/kaggle/input") / DATASET_SLUG
if not DATASET_INPUT_DIR.is_dir():
    raise RuntimeError(
        f"Dataset '{DATASET}' não está anexado como Input deste notebook "
        f"(esperado em {DATASET_INPUT_DIR}). No painel direito do Kaggle: "
        f"Add Input -> Datasets -> busque '{DATASET}' -> Add."
    )

print(f"[INFO] Dataset montado (somente leitura) em: {DATASET_INPUT_DIR}")

In [ ]:
# Sincroniza sempre o repositório como fonte da verdade e copia scripts para execução
import importlib
import shutil

if REPO_DIR.exists():
    print("[INFO] Atualizando repositório via git pull...")
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)
else:
    print("[INFO] Clonando repositório...")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)

# Copiar scripts atualizados do checkout garantindo que versões órfãs sejam eliminadas
if SCRIPTS_DIR.exists():
    shutil.rmtree(SCRIPTS_DIR)
shutil.copytree(REPO_DIR / "scripts", SCRIPTS_DIR)

if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

importlib.invalidate_caches()
for module_name in ("comfyui_setup", "ngrok_tunnel", "gpu_detect", "kaggle_drive_sync"):
    module = sys.modules.get(module_name)
    if module is not None:
        importlib.reload(module)

print("[INFO] Scripts atualizados disponíveis:", sorted(p.name for p in SCRIPTS_DIR.glob("*.py")))

In [ ]:
# Detecção e validação de GPU NVIDIA, depois do checkout dos scripts
from gpu_detect import detect_gpu
GPU_INFO = detect_gpu()
print(json.dumps(GPU_INFO, indent=2, ensure_ascii=False))
if not GPU_INFO.get("has_gpu"):
    raise RuntimeError("GPU NVIDIA não detectada. Ative Accelerator → GPU no Kaggle antes de continuar.")

print("=== GPU / DISCO ===")
subprocess.run(["nvidia-smi"], check=False)
subprocess.run(["df", "-h", str(WORKDIR)], check=False)

In [ ]:
# Configuração do Google Drive (rclone + service account) SOMENTE para persistência/backup
from kaggle_drive_sync import get_drive_path, setup_rclone_kaggle, test_drive_connection

print("=" * 60)
print("CONFIGURANDO GOOGLE DRIVE (PERSISTÊNCIA/BACKUP)")
print("=" * 60)

DRIVE_AVAILABLE = False
try:
    test_res = test_drive_connection(drive_base=DRIVE_BASE, env="kaggle")
    if test_res["status"] == "pass":
        DRIVE_AVAILABLE = True
        DRIVE_PATH = Path(test_res["drive_path"])
        print(f"[INFO] Google Drive pronto para sync em: {DRIVE_PATH}")
    else:
        print(f"[WARN] Google Drive não pôde ser montado: {test_res.get('error')}")
except Exception as e:
    print(f"[WARN] Falha ao configurar Google Drive: {e}")
    print("[INFO] O ComfyUI funcionará normalmente gerando no SSD local.")


In [ ]:
# Provisiona tmpfs e instala/atualiza ComfyUI + custom nodes com output isolado em RAM
from comfyui_setup import setup_comfyui

CUSTOM_NODES = [
    "cubiq/ComfyUI_essentials",
    "lbouaraba/comfyui-krea2edit",
]

# Diretórios tmpfs com permissão total ANTES de qualquer I/O do servidor
for shm_dir in (SHM_OUTPUT, SHM_TEMP):
    shm_dir.mkdir(parents=True, exist_ok=True)
    os.chmod(shm_dir, 0o777)
print(f"[INFO] tmpfs provisionado: {SHM_OUTPUT} | {SHM_TEMP}")
subprocess.run(["df", "-h", "/dev/shm"], check=False)

setup_comfyui(
    comfyui_dir=COMFYUI_DIR,
    models_dir=MODELS_DIR,
    custom_nodes=CUSTOM_NODES,
    output_dir=SHM_OUTPUT,
    additional_model_roots=[("dataset_models", DATASET_INPUT_DIR)],
)
print(f"[INFO] ComfyUI configurado com output em tmpfs (RAM volátil): {SHM_OUTPUT}")
print(f"[INFO] Modelos locais (graváveis) em: {MODELS_DIR}")
print(f"[INFO] Modelos do Dataset lidos diretamente de: {DATASET_INPUT_DIR} (somente leitura, sem cópia)")

In [ ]:
# Inicia/reutiliza ComfyUI com isolamento de I/O em tmpfs, valida a API e só então abre o ngrok
from comfyui_setup import start_comfyui_runtime

COMFYUI_PORT = 8188

# Reassert defensivo: diretórios tmpfs com permissão total imediatamente antes de subir o servidor
for shm_dir in (SHM_OUTPUT, SHM_TEMP):
    shm_dir.mkdir(parents=True, exist_ok=True)
    os.chmod(shm_dir, 0o777)

runtime = start_comfyui_runtime(
    comfyui_dir=COMFYUI_DIR,
    host="127.0.0.1",
    port=COMFYUI_PORT,
    output_dir=SHM_OUTPUT,
    extra_args=["--temp-directory", str(SHM_TEMP)],
    enable_ngrok=True,
)

if not runtime["health"]:
    raise RuntimeError(f"ComfyUI não ficou saudável. Log: {runtime['log_path']}")

if runtime["reused_existing"]:
    print("[WARN] Processo ComfyUI existente reutilizado — confirme que ele foi iniciado")
    print("[WARN] com --output-directory/--temp-directory em /dev/shm. Em dúvida, reinicie o kernel.")

print("=" * 60)
print("COMFYUI READY — ZERO-DISK OUTPUT")
print(f"Local : {runtime['local_url']}")
print(f"Public: {runtime['public_url'] or '(ngrok indisponível)'}")
print(f"Output: {SHM_OUTPUT} (tmpfs)")
print(f"Temp  : {SHM_TEMP} (tmpfs)")
print(f"GPU   : {GPU_INFO.get('gpu_count', 0)} GPU(s)")
print("=" * 60)
if runtime["proc"] is not None:
    print("PID:", runtime["proc"].pid)
else:
    print("PID: processo existente reutilizado")

In [ ]:
# Gerenciamento de ciclo de vida de artefatos:
# polling da API -> coleta em /dev/shm -> zip em /kaggle/working -> purga da RAM volatil
import gc
import shutil
import urllib.request
import zipfile

COMFYUI_API = f"http://127.0.0.1:{COMFYUI_PORT}"

def _api_get(path: str) -> dict:
    with urllib.request.urlopen(f"{COMFYUI_API}{path}", timeout=10) as resp:
        return json.loads(resp.read().decode("utf-8"))

def wait_queue_empty(poll_s: float = 3.0, timeout_s: float = 7200.0) -> None:
    """Bloqueia ate a fila do ComfyUI drenar por completo (running=0, pending=0).

    Um prompt so sai de queue_running depois que todos os outputs foram
    gravados -- drenagem da fila == conclusao exata da geracao.
    """
    start = time.time()
    while True:
        queue = _api_get("/queue")
        running = len(queue.get("queue_running", []))
        pending = len(queue.get("queue_pending", []))
        elapsed = int(time.time() - start)
        print(f"[POLL] running={running} pending={pending} elapsed={elapsed}s   ", end="\r")
        if running == 0 and pending == 0:
            print(f"\n[INFO] Fila drenada apos {elapsed}s. Geracao concluida.")
            return
        if elapsed > timeout_s:
            raise TimeoutError(f"Fila nao drenou em {timeout_s}s -- verifique o log do ComfyUI.")
        time.sleep(poll_s)

def report_history_failures() -> None:
    """Audita /history em busca de jobs com erro antes da compactacao."""
    history = _api_get("/history")
    failures = 0
    for prompt_id, entry in history.items():
        status = entry.get("status", {})
        if status.get("status_str") == "error" or status.get("completed") is False:
            failures += 1
            print(f"[WARN] Job com falha: prompt_id={prompt_id} status={status.get('status_str')}")
    if failures == 0:
        print(f"[INFO] Historico integro: {len(history)} job(s), nenhuma falha.")

def collect_and_zip(shm_output=SHM_OUTPUT, zip_path=SECURE_ZIP) -> int:
    """Compacta todos os artefatos do tmpfs no unico arquivo persistente."""
    files = [p for p in sorted(Path(shm_output).rglob("*")) if p.is_file()]
    if not files:
        print("[WARN] Nenhum artefato em /dev/shm para compactar.")
        return 0
    zip_path = Path(zip_path)
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
        for f in files:
            zf.write(f, arcname=str(f.relative_to(shm_output)))
    size_mb = zip_path.stat().st_size / (1024 ** 2)
    print(f"[INFO] {len(files)} artefato(s) -> {zip_path} ({size_mb:.1f} MB, ZIP_DEFLATED)")
    return len(files)

def purge_shm(*dirs) -> None:
    """Remocao compulsoria do conteudo tmpfs + coleta de lixo do interprete."""
    for d in dirs:
        d = Path(d)
        if d.exists():
            shutil.rmtree(d, ignore_errors=True)
        d.mkdir(parents=True, exist_ok=True)
        os.chmod(d, 0o777)
    gc.collect()
    print(f"[INFO] Purga de RAM volatil concluida: {[str(d) for d in dirs]}")
    subprocess.run(["df", "-h", "/dev/shm"], check=False)

# Fluxo estrito: polling -> auditoria -> compactacao -> purga
wait_queue_empty()
report_history_failures()
collect_and_zip()
purge_shm(SHM_OUTPUT, SHM_TEMP)

In [ ]:
# Validacao de workflow: no 9 (CLIPLoader) deve estar configurado com type "krea2"
WORKFLOW_JSON = WORKDIR / "lustify_simple_t2i.json"  # ajuste o path se o JSON vier do dataset

def validate_workflow_node(path, node_id="9", expected_type="krea2"):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Workflow nao encontrado: {path}")
    wf = json.loads(path.read_text(encoding="utf-8"))

    node = None
    # Formato API (prompt export): {"9": {"class_type": ..., "inputs": {...}}}
    if node_id in wf and isinstance(wf[node_id], dict):
        node = wf[node_id]
    # Formato UI (graph export): {"nodes": [{"id": 9, "type": ...}]}
    elif isinstance(wf.get("nodes"), list):
        node = next((n for n in wf["nodes"] if str(n.get("id")) == node_id), None)

    if node is None:
        raise KeyError(f"No {node_id} ausente em {path.name}")

    actual = node.get("class_type") or node.get("type")
    if actual != expected_type:
        raise ValueError(
            f"No {node_id} com type={actual!r}; esperado {expected_type!r}. "
            f"Re-exporte o workflow com o CLIPLoader krea2."
        )
    print(f"[OK] No {node_id} ({path.name}): type={actual!r} -- workflow compativel")
    return True

validate_workflow_node(WORKFLOW_JSON)

In [ ]:
%%bash
# Validação operacional do isolamento tmpfs -- execute durante/apos a geracao
echo "=== USO DO TMPFS (/dev/shm) ==="
df -h /dev/shm
echo
echo "=== FOOTPRINT DOS DIRETORIOS VOLATEIS ==="
du -sh /dev/shm/comfy_ui_output /dev/shm/comfy_ui_temp 2>/dev/null || echo "(diretorios ainda nao criados)"
echo
echo "=== AUDITORIA ZERO-DISK: nenhuma imagem deve existir em /kaggle/working ==="
find /kaggle/working -type f \( -iname "*.png" -o -iname "*.jpg" -o -iname "*.jpeg" -o -iname "*.webp" -o -iname "*.latent" \) 2>/dev/null | head -20
echo "(vazio acima = garantia de nao-persistencia OK)"
echo
echo "=== UNICO ARTEFATO PERSISTENTE ESPERADO ==="
ls -lh /kaggle/working/output_secure.zip 2>/dev/null || echo "(output_secure.zip ainda nao gerado)"

In [ ]:
# Push inicial condicional: SOMENTE logs. Artefatos de geracao vivem em tmpfs e seguem
# o fluxo output_secure.zip (celula de ciclo de vida) -- nunca passam pelo sync de Drive.
from kaggle_drive_sync import sync_outputs

log_file = COMFYUI_DIR / "comfyui.log"
has_logs = log_file.exists() and log_file.stat().st_size > 0

if has_logs and DRIVE_AVAILABLE:
    try:
        sync_res = sync_outputs(action="push", categories=["logs"], local_outputs=SHM_OUTPUT, drive_base=DRIVE_BASE, env="kaggle")
        print(f"[INFO] Push de logs concluido: {sync_res['synced']} enviado(s), {sync_res['skipped']} inalterado(s)")
    except Exception as e:
        print(f"[WARN] Push de logs nao pôde ser executado: {e}")
else:
    print("[INFO] Sem log para envio ou Drive indisponivel. Pronto para gerar!")
print("[INFO] Imagens geradas NAO residem em /kaggle/working -- zero-disk output ativo.")

## Próximo passo

O ComfyUI está rodando com **isolamento de I/O em tmpfs** — nenhum artefato de geração toca `/kaggle/working`.

Os modelos do Dataset são lidos diretamente do Input montado em `/kaggle/input/<slug>` (somente leitura, sem download).

Fluxo operacional:
1. **Gere** normalmente pela interface ou API (outputs caem em `/dev/shm/comfy_ui_output`).
2. **Execute a célula de ciclo de vida** (polling → `output_secure.zip` → purga) após cada batch de geração.
3. **Baixe** apenas `/kaggle/working/output_secure.zip` pelo painel de output do Kaggle.
4. Valide o workflow antes de enfileirar com a célula de verificação do nó 9 (`lustify_simple_t2i.json`).
5. Monitore o footprint de RAM com a célula de validação (`df -h /dev/shm`).

Para sincronizar **modelos** e logs com o Google Drive, abra e execute o notebook manual:
**`09_sync_outputs.ipynb`**

## [Opcional] Civitai → Local: adicionar modelo faltante rapidamente

Célula **independente do fluxo principal** (não roda no Run All por padrão — execute manualmente).
Use quando faltar algum modelo no ComfyUI que já está de pé: baixa direto da Civitai por AIR/URL
(mesma fila interativa do `colab_transfer/00_master_pipeline.ipynb`) e salva direto em
`MODELS_DIR/<categoria>/` — sem staging, sem manifest e sem publicar no Kaggle Dataset.

Pré-requisitos:
- Já ter rodado a Célula 4 (sync do repositório) nesta sessão, para `kaggle_dataset_manager`
  estar em `sys.path`.
- Secret `CIVITAI_TOKEN` (ou `CIVITAI_API_KEY`) configurado — mesmo token usado no Colab.

Fluxo desta célula: fila AIR/URL até `done` → resolve metadados do lote → se houver checkpoint,
pergunta uma única vez o destino (`checkpoints/` ou `diffusion_models/`) → baixa tudo em sequência
direto para a pasta de modelos já usada pelo ComfyUI. Depois de rodar, clique em **Refresh** no
Manager do ComfyUI (ou re-execute a Célula 8) para o modelo novo aparecer na interface.

In [ ]:
# ============================================================
# CIVITAI -> LOCAL: adição rápida de modelo (fora do fluxo principal do 08)
# Reaproveita a mesma fila AIR/URL e resolução de metadados do
# colab_transfer/00_master_pipeline.ipynb (scripts/kaggle_dataset_manager.py),
# só que sem staging/manifest/publicação: o destino final já é
# MODELS_DIR/<categoria>/, a mesma hierarquia que o ComfyUI já lê neste notebook.
# ============================================================

# kaggle_dataset_manager.py resolve um dataset padrão no import (não usado aqui,
# já que esta célula não publica nada) — garante que os Secrets já lidos por
# _get_secret também estejam em os.environ, evitando falha no import.
os.environ.setdefault("KAGGLE_USERNAME", _get_secret("KAGGLE_USERNAME") or "")
os.environ.setdefault("KAGGLE_DATASET_NAME", _get_secret("KAGGLE_DATASET_NAME") or "")

from kaggle_dataset_manager import (
    CATEGORIES,
    collect_input_queue,
    download_resolved_queue,
    queue_contains_checkpoint,
    resolve_queue_metadata,
)

# Garante a mesma hierarquia de pastas já criada pelo setup do ComfyUI (Célula 7),
# mesmo que esta célula seja executada antes dele.
for _cat in CATEGORIES:
    (MODELS_DIR / _cat).mkdir(parents=True, exist_ok=True)

CIVITAI_TOKEN = _get_secret("CIVITAI_TOKEN") or _get_secret("CIVITAI_API_KEY")
if not CIVITAI_TOKEN:
    raise RuntimeError(
        "CIVITAI_TOKEN não encontrado nos Secrets do Kaggle. "
        "Adicione o Secret CIVITAI_TOKEN (mesmo usado no Colab) antes de rodar esta célula."
    )

print("=" * 60)
print("CIVITAI -> LOCAL (adição rápida de modelo)")
print(f"Destino: {MODELS_DIR}")
print("=" * 60)

# 1) Fila AIR/URL -- digite 'done' para fechar a lista (mesmo loop do 00_master_pipeline.ipynb)
_pending = collect_input_queue()

if not _pending:
    print("[INFO] Nenhum item informado; nada a baixar.")
else:
    # 2) Resolve metadados do lote inteiro ANTES de baixar (nenhum arquivo tocado ainda)
    _resolved = resolve_queue_metadata(_pending, CIVITAI_TOKEN)
    if not _resolved:
        print("[ERROR] Nenhum item válido resolvido; nada a baixar.")
    else:
        # 3) Se houver checkpoint no lote, pergunta o destino uma única vez (não por item)
        _checkpoint_destination = None
        if queue_contains_checkpoint(_resolved):
            _choice = input("Checkpoint: 1=checkpoints/ 2=diffusion_models/: ").strip().lower()
            _checkpoint_destination = {"1": "checkpoints", "2": "diffusion_models"}.get(_choice, _choice)
            print(f"[INFO] Destino de checkpoints deste lote: {_checkpoint_destination}")

        # 4) Download sequencial direto para MODELS_DIR/<categoria>/ -- sem nenhuma pausa nova
        _downloaded = download_resolved_queue(
            _resolved,
            staging_dir=MODELS_DIR,
            token=CIVITAI_TOKEN,
            checkpoint_destination=_checkpoint_destination,
        )

        print(f"\n[SUCCESS] {len(_downloaded)} arquivo(s) adicionados em {MODELS_DIR}")
        for _item in _downloaded:
            print(f"  + {_item.path} ({_item.size / (1024**3):.2f} GB)")
        print(
            "\n[INFO] Modelo(s) já estão no disco persistente do ComfyUI. "
            "Clique em Refresh no Manager (ou re-rode a Célula 8) para aparecerem na interface."
        )